##Model Training##
This notebook trains a teacher model, distills it into a student model, and exports the final model artifacts.  
The output includes the trained **.h5 model**, **.tflite model** for profiling.

##Setup Project##
- Add the directory containing helper scripts for training a custom model.
- Install Dependencies
- Restart Python environment

In [0]:
!curl -LsSf https://astral.sh/uv/install.sh | sh
!uv pip install silabs-mltk audiomentations noisereduce pyloudnorm librosa tensorflow scikit-learn tensorflow-model-optimization vosk

import sys
import os

sys.path.append(os.path.abspath("."))

dbutils.library.restartPython()

## Export Configuration

In [0]:
WORKSPACE_DIR = "/Workspace/Users/raansari@silabs.com"

MODEL_PATH = f"{WORKSPACE_DIR}/vosk-model-small-en-us-0.15"
AUDIO_PATH = "/Volumes/mlops_dev/default/audio_raw"

WORKSPACE_MODELS_DIR = f"{WORKSPACE_DIR}/models"
Teacher_H5 = f"{WORKSPACE_MODELS_DIR}/custom_model_teacher.h5"
DataRoot = AUDIO_PATH

###Preprocessing the data
Cleaning the raw audio data for removing false positives

In [0]:
import vosk
import os
import sys

from vosk_preprocessing import run_refinement

print(f"--- Starting Offline Refinement on {AUDIO_PATH} ---")

# Run refinement and rename files immediately
results = run_refinement(AUDIO_PATH, MODEL_PATH, auto_rename=True)

if results:
    print(f"\nSUCCESS: Refined {len(results)} files. Updating metadata table...")

    for res in results:
        # 2. Update the metadata table in Unity Catalog
        # We update file_name, file_path, and class_label based on the refinement
        query = f"""
            UPDATE mlops_dev.default.stream_audio_metadata 
            SET file_name = '{res["new_name"]}', 
                file_path = '{res["new_path"]}', 
                class_label = '{res["refined_label"]}'
            WHERE file_name = '{res["original_name"]}'
        """
        try:
            # Execute Spark SQL update
            spark.sql(query)
            print(f" [SQL OK] Updated metadata for {res['original_name']}")
        except Exception as sql_err:
            print(
                f" [SQL ERR] Failed to update table for {res['original_name']}: {sql_err}"
            )

    print("\n--- Refinement and Metadata Sync Complete ---")
else:
    print("\nNo 'unknown' files were successfully refined at this time.")

##Train Teacher Model##

Trains the teacher model by **loading a base model if it exists**, otherwise **starting from scratch**. The `mltk_core` pipeline automatically resumes from an existing `custom_model_teacher.h5` or initializes a new model when none is found.

In [0]:
# Train Teacher Model
import os

os.environ["TRAIN_TEACHER"] = "1"

import mltk.core as mltk_core
from custom_model import my_model, prepare_teacher_or_student_model

prepare_teacher_or_student_model(train_teacher=True)

base_model_path = f"{WORKSPACE_DIR}/models/Custom_model_teacher.h5"
if os.path.exists(base_model_path):
    print(f"Resuming training from: {base_model_path}")
    results = mltk_core.train_model(my_model, clean=False, weights=base_model_path)
    _, teacher_accuracy = results.get_best_metric()
else:
    print("Starting training from scratch...")
    results = mltk_core.train_model(my_model, clean=True)
    _, teacher_accuracy = results.get_best_metric()

print(results)

##Store Teacher accuracy##

In [0]:
# creating .txt file in /tmp folder to store teacher accuracy
path = f"{WORKSPACE_DIR}/tmp/teacher_accuracy.txt"
os.makedirs(os.path.dirname(path), exist_ok=True)

with open(path, "w") as f:
    f.write(str(teacher_accuracy))

## Train Student Model

Trains the student model by **loading a previous student checkpoint if available**, otherwise **starting from scratch**, and uses the fine‑tuned teacher model for knowledge distillation.

In [0]:
# Train Student Model
import os

os.environ["TRAIN_TEACHER"] = "0"

import mltk.core as mltk_core
from custom_model import my_model, prepare_teacher_or_student_model

prepare_teacher_or_student_model(train_teacher=False)

base_model_path = f"{WORKSPACE_DIR}/models/Custom_model.h5"
if os.path.exists(base_model_path):
    print(f"Resuming training from: {base_model_path}")
    results = mltk_core.train_model(my_model, clean=False)
    _, student_accuracy = results.get_best_metric()
else:
    print("Starting training from scratch...")
    results = mltk_core.train_model(my_model, clean=True)
    _, student_accuracy = results.get_best_metric()

print(results)

##Store Student accuracy##

In [0]:
# creating .txt file in /tmp folder to store student accuracy
path = f"{WORKSPACE_DIR}/tmp/student_accuracy.txt"
os.makedirs(os.path.dirname(path), exist_ok=True)

with open(path, "w") as f:
    f.write(str(student_accuracy))

##Locate and Copy Model Artifacts##
This step searches for trained model artifacts inside driver nodes, identifies the correct `custom_model` folder, and copies the generated `.h5` and `.tflite` files into your Workspace for easy access.

In [0]:
import glob, os, pprint, shutil

# 1) Find all potential driver homes
driver_homes = [d.rstrip("/") for d in glob.glob("/home/spark-*/")]
if not driver_homes:
    raise RuntimeError("No /home/spark-*/ found on this cluster")

# 2) For each driver, look for a non-empty custom_model folder
candidates = []
for dh in driver_homes:
    cm_dir = f"{dh}/.mltk/models/custom_model"
    if os.path.isdir(cm_dir):
        try:
            files = os.listdir(cm_dir)
        except Exception:
            files = []
        candidates.append((cm_dir, files))

print("MLTK candidate dirs:")
pprint.pp(candidates)

# 3) Choose the one that actually has artifacts (h5 or tflite)
src_dir = None
for cm_dir, files in candidates:
    if any(f.endswith((".h5", ".tflite")) for f in files):
        src_dir = cm_dir
        break

if not src_dir:
    raise RuntimeError(
        "No custom_model artifacts found. Re-run STUDENT training, then run this cell again."
    )

# 4) Copy selected artifacts to Workspace Files
# Creating a folder in workspace files to store the models Eg: models

dst_dir = f"{WORKSPACE_DIR}/models"
os.makedirs(dst_dir, exist_ok=True)

for name in [
    "custom_model.h5",
    "custom_model.tflite",
    "custom_model.tflite.summary.txt",
]:
    src = os.path.join(src_dir, name)
    if os.path.exists(src):
        print("Copying:", src, "->", os.path.join(dst_dir, name))
        shutil.copy2(src, os.path.join(dst_dir, name))
    else:
        print("[WARN] Missing:", src)